# M23 — M22 + window-phase jitter on all four platforms, submitted

## M23 — window-phase jitter: the same recordings, cut into windows at a different place

**What moves.** One key on top of M22 (the 0.36871 submission, val 0.2288):
`augment.timescale.phase_p = 1.0` on all four platforms. M22's drone speed-ups stay
exactly as they were (same draws, same seed).

**What it does.** Every recording is cut into one-second windows at fixed places
(frames 0-199, 200-399, ...), so every epoch the model has seen *exactly the same*
windows — and a drone flight is a single chunk, so it is literally the same sample
every epoch. This op starts each chunk's window grid a random 1-199 frames later,
fresh every epoch. Nothing is synthesised: the six channels are the recording's own
frames, the labels are the window means of the recorded per-frame velocity, and the
gravity channels are the run's cached estimate at those frames. It is what the
sensor would have produced had the recording started a moment later.

**Why.** The model memorises its training drones: the M22 model scores **0.096 m/s**
on its own training drone flights and **0.376** on val's, a 3.9× gap (car 2.6×,
dog 2.3×, human 1.6×). Two retrains of M22 disagree by **0.24 m/s per drone window**
on val and **0.40** on test, and averaging two of them cuts drone error 6%. Drone's
remaining error is mostly *variance* — too few distinct samples for what the model
can memorise — not a missing mechanism: a per-flight constant rotation, a per-flight
calibration and the rotor-drag cue each fail to recover any of it
(`.claude/agentTests/2026-09-19_drone-after-m22/`). More distinct samples is the
textbook remedy, and adding val's 48 flights (D9b + M22) already moved public
0.36871 → 0.35611.

**Proved before any GPU time.** `mySolution/tests/test_phase.py` (6 checks): at
shift 0 the re-cut path reproduces the plain sample on all four platforms (channels
0, labels 1.9e-06 m/s); at a real shift the raw channels are bit-exact, the labels
match the raw per-frame velocity to 1e-5, and M22's speed draws are unchanged.
Loader cost: 2.9 ms per chunk warm (measured on the laptop), ~0.7 GB of per-frame
truth held in memory.

**What to expect.** Val should improve, drone most **[expected]**; the public board
decides against 0.36871 (this arm) or 0.35611 (its train+val twin).

---

### Before you run

1. **Attach the competition data.** Right panel -> *Add Input* -> *Competitions*
   -> **TartanIMU Challenge: Multi-Platform Inertial Odometry**. Nothing else
   needs attaching; all the code is inside this notebook.
2. **Turn the GPU on.** Right panel -> *Accelerator* -> **GPU T4 x2** (only one
   is used) or **P100**.
3. **Turn the internet off.** Nothing here downloads anything.

### What it produces

`/kaggle/working/proj/mySolution/runs/` in exactly the layout the laptop
writes — one directory per run with `config.json`, `env.json`, `metrics.json`,
`history.json`, `model.pt` and `predictions/submission_val.csv` and `predictions/submission_test.csv`,
plus the append-only `results.jsonl`. The last cell zips it to
`/kaggle/working/tartanimu_runs.zip`.

**Bringing it home:** unzip, copy the run directories into `mySolution/runs/`,
and append the lines of the Kaggle `results.jsonl` to the local one. They will
not collide: `run.device` is part of the config hash on purpose, so a `cuda` run
and an `mps` run of the same science are different hashes and different
directory names.

**The file to submit** is `predictions/submission_test.csv` inside the one run directory this notebook writes — copy it out, or download it straight from the zip, and upload it to the competition's submission page.

### On restarts

Every stage is idempotent. If the session dies, re-running the whole notebook picks up where it left off: the caches rebuild in about two minutes, and the one seed is skipped if it is already in `results.jsonl`. **Kaggle only keeps
`/kaggle/working` across sessions if you *Save Version*** — for a mid-run
rescue, download the zip instead.

## 1. Knobs

Everything you might want to change is here and nowhere else.

In [1]:
# --- the science -----------------------------------------------------------
REFERENCE  = "m9_film"    # the reference config this arm sits on top of
INPUT_REPR = "body_grav"       # nine channels, as M22 -- only augment.timescale.phase_p moves
GRAVITY_SOURCE = "filter"
                          # where the three gravity channels are computed:
                          #   "filter"  one pass of the complementary filter over
                          #             each WHOLE trajectory, cached before
                          #             training. Every run up to M7 used this.
                          #   "chunk"   the same filter, re-run inside the loader
                          #             on THIS CHUNK's frames and nothing else,
                          #             so the input is a pure function of the
                          #             sample. Costs a measured +0.06 deg on a
                          #             channel whose own error is 2.86 deg.
                          #   "truth"   an ORACLE: the true down-direction from
                          #             the ground-truth quaternion. Needs
                          #             DIAGNOSTIC_ONLY; can never predict test.
                          # Ignored when INPUT_REPR is "body" -- no gravity channels.
DIAGNOSTIC_ONLY = False  # True only for an oracle that reads ground truth into
                          # the model. Stamped into every result line, so the run
                          # can never be quietly compared against a real one.
GRAVITY_ESTIMATOR = None  # None = the causal complementary filter every run
                          # through M18 used (prep.gravity.estimator absent)
TRUTH_NOISE_DEG = 0.0     # M20 only: tilt error added to the truth oracle
TRUTH_NOISE_TAU_S = 3.0
ATTITUDE_BRANCH = None    # M21 only: AirIO's separate attitude branch
TRAINVAL = None           # D9: val stays out of training, as in every run before it
MILESTONE  = "M23"
TAG        = "m23-phase"       # run names become f"{TAG}-seed{S}"
SEEDS      = [42] # one seed -- a submission is one shared set of
                          # weights, not an ensemble of three
EXCLUDE_SEAL = False  # False: this run also trains on the 61
                          # sealed trajectories. Correct ONLY for a real
                          # submission -- there is no later honest check
                          # left to spend on this exact model.
NOTES      = "M23: M22 (timescale_drone, p 0.5, s 1.0-1.4) + window-phase jitter (grid shifted 1-199 frames, all platforms, every chunk). SUBMISSION run: one seed, trained on train+seal, predicts the test split."
AUGMENT_RECIPE = None
                          # None = train on the data as recorded, which is what
                          # every run before M17 did. A recipe name switches on
                          # the re-mounting augmentation in src/data/augment.py:
                          # the same motion, recorded by a sensor bolted on in a
                          # different orientation. Input AND target are rotated
                          # together, so nothing physically impossible is taught
                          # -- mySolution/tests/test_augment.py is the gate.
                          #   "remount_v1"     yaw jitter (drone: the full
                          #                    circle, the other three 10-15 deg)
                          #                    plus 5 deg of tilt everywhere
                          #   "remount_drone"  drone yaw only, nothing else
                          #   "remount_mirror" remount_v1 plus left/right mirrors
                          # Costs nothing: the complementary filter is exactly
                          # equivariant, so the gravity channels are rotated in
                          # place rather than recomputed (.claude/augmentation.md).

# --- M22: time scaling -- the same drone flights, flown faster ---------------
TIMESCALE = {'recipe': 'timescale_drone', 'phase_p': {'car': 1.0, 'dog': 1.0, 'drone': 1.0, 'human': 1.0}}
                          # "timescale_drone": each epoch, half the DRONE flights
                          # are played back s times faster, s log-uniform in
                          # [1.0, 1.4]. Below 8 Hz the motion is scaled (linear
                          # acceleration x s^2, gyro x s, velocity target x s);
                          # above 8 Hz the vibration is kept at its own speed;
                          # gravity is untouched; the three gravity channels are
                          # re-estimated on the new IMU. Car, dog, human, val and
                          # test are never touched (src/data/timescale.py).
                          # To try other magnitudes, add keys, e.g.
                          #   "s_range": {"car": [1, 1], "dog": [1, 1],
                          #               "drone": [1.0, 1.3], "human": [1, 1]},
                          #   "p": {"car": 0, "dog": 0, "drone": 0.5, "human": 0}
                          # A different number is a different arm: change TAG too.
                          # None switches it off (= the 0.39119 control).
                          # "phase_p" (M23), per platform: the probability that
                          # a chunk's window grid starts 1-199 frames later --
                          # the recording's own frames and labels, re-cut.
MODEL_OVERRIDES = None    # M24 only: a model key moved (e.g. the encoder)
GRAVITY_HEAD = False      # M15's auxiliary gravity head; not part of this arm
RUN_CONTROL  = False      # no control run in this arm

# --- the session -----------------------------------------------------------
FORCE_CACHE_REBUILD = True  # rebuild every Tier 0 cache from scratch, ignoring
                            # any copy already on disk. The first T4 control
                            # attempt trained on a STALE gravity cache that
                            # /kaggle/temp had kept from an earlier run and
                            # scored 0.93 instead of 0.25. A gravity run must
                            # never trust a persisted cache. Costs ~90 s.
SMOKE_EPOCHS   = None     # set to 2 for a ~4-minute end-to-end rehearsal
BUILD_TEST_CACHE = True  # True also caches the test split, so
                          # the submission cell below can predict it.
SUBMIT_TEST      = True # after training, reload the checkpoint and
                          # predict the competition test split, writing
                          # submission_test.csv (see cell 8 below)
CUDNN_BENCHMARK = True    # picks the fastest conv algorithm per shape. Shapes
                          # here are constant (one bucket of 64), so the choice
                          # is made once and the run stays reproducible.
HEARTBEAT_SECONDS = 30    # how often the in-place step counter is allowed
                          # through. The training loop redraws it four times a
                          # second, which is right for a terminal and wrong for
                          # a notebook: nine hours of that is ~500,000 writes
                          # and a saved notebook nobody can open. Epoch lines
                          # are never throttled.

# --- where things live -----------------------------------------------------
WORK_ROOT  = "/kaggle/working/proj"    # the repo layout: WORK_ROOT/{mySolution,TartanIMU}
CACHE_ROOT = "/kaggle/temp/tartanimu"  # ~500 MB of Tier 0 cache, NOT saved as output

print(f"{TAG}: {REFERENCE} with data.input_repr={INPUT_REPR!r}, "
      f"prep.gravity.source={GRAVITY_SOURCE!r}, exclude_seal={EXCLUDE_SEAL}, "
      f"seeds {SEEDS}")
if not EXCLUDE_SEAL:
    print("*** TRAINS ON THE SEALED HOLDOUT TOO (EXCLUDE_SEAL = False) -- this is a "
          "submission run. Its val score is not comparable to any run that kept "
          "the seal shut. ***")
print(f"TIME SCALING: {TIMESCALE} -- drone training flights only; val and test are never transformed")
if DIAGNOSTIC_ONLY:
    print("*** DIAGNOSTIC ONLY: an oracle. Reads ground truth; can never be submitted. ***")
if FORCE_CACHE_REBUILD:
    print("Tier 0 caches will be rebuilt from scratch (FORCE_CACHE_REBUILD = True)")
if SMOKE_EPOCHS:
    print(f"*** SMOKE TEST: {SMOKE_EPOCHS} epochs. These runs are not results. ***")

m23-phase: m9_film with data.input_repr='body_grav', prep.gravity.source='filter', exclude_seal=False, seeds [42]
*** TRAINS ON THE SEALED HOLDOUT TOO (EXCLUDE_SEAL = False) -- this is a submission run. Its val score is not comparable to any run that kept the seal shut. ***
TIME SCALING: {'recipe': 'timescale_drone', 'phase_p': {'car': 1.0, 'dog': 1.0, 'drone': 1.0, 'human': 1.0}} -- drone training flights only; val and test are never transformed
Tier 0 caches will be rebuilt from scratch (FORCE_CACHE_REBUILD = True)


## 2. Unpack the source tree

52 files — `src/`, the committed seal, the vendored scorer, and the three
`repro/` modules the scorer path imports — packed as a 198 KB base64 blob by
`notebooks/build_kaggle_notebook.py`. No data, no caches, no weights.

The archive reproduces the repository layout, `TartanIMU/` next to
`mySolution/`, because `src/paths.py` derives every path from its own location
and would not find the scorer otherwise.

In [2]:
import base64, hashlib, io, os, shutil, sys, tarfile, time
from pathlib import Path

SOURCE_SHA256 = "5845f6cef0aabe67286db1b89bea3a0cb96e568fb332778ced839f12480d3556"
SOURCE_BLOB = "\
..."

raw = base64.b64decode(SOURCE_BLOB)
got = hashlib.sha256(raw).hexdigest()
assert got == SOURCE_SHA256, f"blob is corrupt: {got} != {SOURCE_SHA256}"

work = Path(WORK_ROOT)
shutil.rmtree(work / "mySolution" / "src", ignore_errors=True)   # always fresh code
work.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as tar:
    # `filter="data"` is the safe default from Python 3.12 and is required from
    # 3.14; older runtimes do not accept the keyword at all.
    try:
        tar.extractall(work, filter="data")
    except TypeError:
        tar.extractall(work)

MYSOLUTION = work / "mySolution"
sys.path.insert(0, str(MYSOLUTION))
n = sum(1 for _ in work.rglob("*") if _.is_file())
print(f"unpacked {n} files into {work}  (sha256 {SOURCE_SHA256[:12]})")

unpacked 52 files into /kaggle/working/proj  (sha256 5845f6cef0aa)


## 3. Point the code at the competition data

`src/paths.py` expects `mySolution/data/raw/{train,val,test,index}`, which is
exactly the layout the competition download has, so the raw directory becomes a
symlink into `/kaggle/input` and nothing else has to know.

**Where Kaggle puts it depends on how it is attached.** A competition lands at
`/kaggle/input/competitions/<slug>`, a user dataset at `/kaggle/input/<slug>`.
The finder searches both depths and identifies the right directory by looking
for `index/train_windows.csv` inside it, rather than by matching a name.

The Tier 0 caches go to `/kaggle/temp` rather than `/kaggle/working`. They are
about 500 MB of derived, rebuildable-in-two-minutes intermediate, and putting
them in the output would make every *Save Version* copy them.

In [3]:
def find_competition_data() -> Path:
    """The attached input directory that looks like the competition download.

    Kaggle mounts competition data one level deeper than a user dataset --
    /kaggle/input/competitions/<slug> against /kaggle/input/<slug> -- so both
    depths are searched rather than one guessed. The test is the index file
    itself, not the directory name, which also makes this work if you ever do
    upload the data as your own dataset under some other name.
    """
    root = Path("/kaggle/input")
    if not root.exists():
        raise SystemExit("/kaggle/input does not exist -- is this running on Kaggle?")
    candidates = sorted(root.glob("*")) + sorted(root.glob("*/*"))
    for c in candidates:
        if (c / "index" / "train_windows.csv").is_file():
            return c
    raise SystemExit(
        "no competition data found under /kaggle/input. Add Input -> Competitions\n"
        "-> TartanIMU Challenge: Multi-Platform Inertial Odometry.\n"
        f"Looked for */index/train_windows.csv and */*/index/train_windows.csv.\n"
        f"Currently attached: {[str(c.relative_to(root)) for c in candidates] or 'nothing'}")

RAW = find_competition_data()
raw_link = MYSOLUTION / "data" / "raw"
raw_link.parent.mkdir(parents=True, exist_ok=True)
if raw_link.is_symlink() or raw_link.exists():
    raw_link.unlink()
raw_link.symlink_to(RAW)

cache_root = Path(CACHE_ROOT)
cache_root.mkdir(parents=True, exist_ok=True)
processed = MYSOLUTION / "data" / "processed"
if processed.is_symlink() or processed.exists():
    (processed.unlink() if processed.is_symlink() else shutil.rmtree(processed))
processed.symlink_to(cache_root)

from src.paths import INDEX, RAW as SRC_RAW, RUNS
import pandas as pd
for split in ("train", "val", "test"):
    df = pd.read_csv(INDEX / f"{split}_windows.csv")
    print(f"{split:>6}: {len(df):>6} windows, {df.traj_id.nunique():>3} trajectories")
print(f"\nraw       {SRC_RAW} -> {RAW}")
print(f"cache     {processed} -> {cache_root}")
print(f"runs      {RUNS}")

 train:  81931 windows, 395 trajectories
   val:  23714 windows,  80 trajectories
  test:  30644 windows,  89 trajectories

raw       /kaggle/working/proj/mySolution/data/raw -> /kaggle/input/competitions/tartan-imu-challenge-iros2026
cache     /kaggle/working/proj/mySolution/data/processed -> /kaggle/temp/tartanimu
runs      /kaggle/working/proj/mySolution/runs


## 4. The environment

Nothing is installed — Kaggle's GPU image already carries torch, numpy and
pandas. This cell only records what it found, because `env.json` in every run
directory will carry the same record and an unexplained shift in a table is
usually a version change.

In [4]:
import torch
from src.utils import env_record

env = env_record()
print(f"python {env['python']}   torch {env['packages']['torch']}   "
      f"numpy {env['packages']['numpy']}   pandas {env['packages']['pandas']}")
print(f"device available: {env['device_available']}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"gpu: {p.name}, {p.total_memory / 1e9:.1f} GB, {torch.cuda.device_count()} visible")
    torch.backends.cudnn.benchmark = bool(CUDNN_BENCHMARK)
    print(f"cudnn.benchmark = {torch.backends.cudnn.benchmark}")
else:
    print("*** NO GPU. Turn the accelerator on, or this will take days. ***")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

python 3.12.13   torch 2.10.0+cu128   numpy 2.0.2   pandas 2.3.3
device available: cuda
gpu: Tesla T4, 15.6 GB, 2 visible
cudnn.benchmark = True


## 5. Build the Tier 0 caches

Four builders, all idempotent — re-running this cell after a restart is a
no-op for whatever already exists. **Except that by default it starts from
nothing:** `FORCE_CACHE_REBUILD` wipes `/kaggle/temp/tartanimu` first, because
the first T4 control attempt trained on a stale gravity cache left over from an
earlier session and scored 0.93 instead of 0.25. Every orientation cache built
here is then checked against ground truth and the cell refuses to continue if
it reads tens of degrees instead of a few.

- **windows** — the flat frame-major fp16 cache. Also asserts, on every build,
  that each published target equals the per-window mean of `vel_body` exactly.
- **splits** — *verifies* rather than draws. It redraws the sealed holdout from
  the data and fails loudly if the draw no longer reproduces the committed
  `configs/splits_v1.json`. That is the check that the data here is the data the
  seal was cut from.
- **segments** — the ATE20 segment bounds. Not read by this arm (the ATE20 loss
  term is off and `snap_to_segments` is off), built anyway because it costs a
  tenth of a second and its absence is a confusing failure later.
- **norm** — the fixed per-channel normalisation constants, measured once over
  train and then frozen. On a six-channel arm the last line prints `ax..gz` and
  no `ux uy uz`.

  **On a nine-channel arm this needs the orientation filter over train, even
  when the run itself estimates gravity per chunk.** That is not a contradiction
  and it is worth being precise about: normalisation constants are a *frozen
  number* measured once, morally part of the weights, so they may depend on
  training data in ways the inference path may not. What matters is that nothing
  at inference reads the cache — the preflight cell below builds the val dataset
  and asserts it holds no trajectory-wide gravity array at all. The two versions
  of the constants would agree anyway: the channels are unit vectors, and
  restarting the filter per chunk moves them by 0.06° at the macro median.

  A **control run** that reads nine channels from the cache (`m9_film`) also
  needs the filter over val, and gets it here — about 35 s for both splits.

  **The ground-truth oracle** (`GRAVITY_SOURCE = "truth"`) needs the filter
  over train only, for the same constants: it reuses `m9_film`'s normalisation
  unchanged, so the one thing that differs between it and `m9_film` is what
  goes into the last three channels. Its own channels are computed in the
  loader from the ground-truth quaternion.

In [5]:
from src.prep import norm as norm_mod
from src.prep import orientation as orientation_mod
from src.prep import segments as segments_mod
from src.prep.orientation import orientation_available
from src.prep import splits as splits_mod
from src.prep import windows as windows_mod

# A gravity run must never inherit a persisted cache. The first attempt at the T4
# control trained against a stale gravity_body.npy that /kaggle/temp had kept
# from an earlier run and scored 0.93 instead of 0.25. Wipe the whole Tier 0
# tree so windows, orientation, norm and segments are all rebuilt together and
# cannot drift out of alignment. The sealed-holdout file lives in the repo, not
# here, so it is untouched -- still just verified below.
if FORCE_CACHE_REBUILD:
    shutil.rmtree(Path(CACHE_ROOT), ignore_errors=True)
    Path(CACHE_ROOT).mkdir(parents=True, exist_ok=True)
    print(f"  wiped {CACHE_ROOT}")

splits_to_build = ["train", "val"] + (["test"] if BUILD_TEST_CACHE else [])

t0 = time.perf_counter()
windows_mod.build_all(splits_to_build, force=FORCE_CACHE_REBUILD)
print(f"  windows   {time.perf_counter() - t0:.1f}s")

t0 = time.perf_counter()
splits_mod.build()                       # verifies the committed seal reproduces
print(f"  splits    {time.perf_counter() - t0:.1f}s")

t0 = time.perf_counter()
for s in ("train", "val"):
    segments_mod.build_split(s, force=FORCE_CACHE_REBUILD)
print(f"  segments  {time.perf_counter() - t0:.1f}s")

import json as _json

from src import config as config_mod
from src.prep.features import needs_gravity
from src.prep.orientation import estimator_key

# Every (input_repr, gravity source, estimator) this session trains -- the
# control run included, read off its resolved config rather than restated as a
# knob. The estimator travels as sorted JSON so the triple stays hashable.
needs = [(INPUT_REPR, GRAVITY_SOURCE, _json.dumps(GRAVITY_ESTIMATOR or {}, sort_keys=True))]
if RUN_CONTROL:
    _c = config_mod.resolve(CONTROL_REFERENCE)
    _cg = dict(_c["prep"].get("gravity", {}))
    needs.append((_c["data"]["input_repr"], _cg.pop("source", "filter"),
                  _json.dumps(_cg, sort_keys=True)))

_built = set()


def _build_gravity(s: str, est) -> None:
    """Build one split's cache for one estimator (None = the default filter), once."""
    key = estimator_key(est)
    if (s, key) in _built:
        return
    _built.add((s, key))
    if est and est.get("gyro") == "denoised":
        # Lever 4: the denoiser is trained here, on train only, deterministically
        # seeded, and saved next to the caches -- the wipe above removed any old one.
        from src.prep import gyro_denoise as _gd
        if not _gd.denoiser_path().exists():
            print("  training the gyro denoiser on train (lever 4) ...", flush=True)
            _meta = _gd.train(device=DEVICE)
            print(f"  gyro denoiser: {_meta['n_parameters']:,} parameters, {_meta['seconds']} s")
            for _p, _d in _gd.rate_report(_gd.load_denoiser()).items():
                print(f"    val {_p:<6} rate error (32-frame means): raw {_d['raw']:.4f}  "
                      f"denoised {_d['denoised']:.4f} rad/s")
    print(f"  orientation ({s}{', ' + key if key else ''}) ...", flush=True)
    orientation_mod.build_split(s, force=FORCE_CACHE_REBUILD, gravity_cfg=est)
    if s == "test":
        # Test carries no ground-truth quaternion, so verify() has nothing to
        # check against. Assert the shape and finiteness instead, which is what
        # would catch a broken build here.
        import numpy as _np
        from src.prep.orientation import load_orientation as _load
        _g = _load("test", key=key)
        _n = _np.linalg.norm(_np.asarray(_g, _np.float64), axis=-1)
        assert _np.isfinite(_n).all() and abs(float(_n.mean()) - 1.0) < 1e-3, \
            f"test gravity cache looks wrong: mean |g| = {float(_n.mean()):.4f}"
        print(f"  gravity (test{', ' + key if key else ''}): {len(_n):,} frames, all "
              f"unit-norm and finite (no ground truth on test, so no angle check)")
        return
    # The check that would have caught the poisoned run in seconds: a misaligned
    # or wrong-signed cache reads tens of degrees here, not single digits. The
    # filter's honest number is ~2 to 3 deg. Run on every split built, train
    # included -- the normalisation constants are measured on train's channels,
    # so a bad train cache poisons every arm, the oracle too.
    chk = orientation_mod.verify(s, key=key)
    print(f"  gravity vs truth ({s}{', ' + key if key else ''}): "
          f"{chk['median_angle_deg']:.2f} deg macro   "
          f"{chk['median_angle_deg_per_platform']}   "
          f"sign_ok={chk['sign_convention_is_correct']}")
    assert chk["sign_convention_is_correct"] and chk["median_angle_deg"] < 10.0, \
        f"{s} gravity cache looks wrong, refusing to train on it: {chk}"


t0 = time.perf_counter()
for repr_, source, est_json in dict.fromkeys(needs):
    est = _json.loads(est_json) or None
    # Every representation whose name the features module lists as needing the
    # filter -- body_grav appends the estimate, and the two `aligned` modes ROTATE
    # by it, so all three need the cache. This used to test `repr_ == "body_grav"`
    # by name, which silently skipped the orientation build for an aligned arm and
    # would have failed it on `norm.load` after the GPU session had already spun up.
    if needs_gravity(repr_):
        # Train with the DEFAULT filter always, to measure the normalisation
        # constants. Those are a frozen constant measured once over train --
        # morally part of the weights -- so they are allowed a training-time
        # dependency the inference path does not have. They stay the default
        # filter's even on an M19 arm, so the ONLY thing an estimator arm changes
        # is the value of the three channels, never how they are scaled.
        _build_gravity("train", None)
        # The run's own estimator, on every split it reads at inference: val
        # when the run reads the cache (source "filter"), and TEST too when this
        # notebook writes a test submission -- without it the submission cell
        # below raises on a missing cache, after the GPU hours are spent. In
        # "chunk" mode the dataset never opens a cache, and the preflight
        # below asserts that.
        if source == "filter":
            for s in ["train", "val"] + (["test"] if SUBMIT_TEST else []):
                _build_gravity(s, est)
    norm_mod.build(repr_, "train", force=FORCE_CACHE_REBUILD)
print(f"  orientation + norm  {time.perf_counter() - t0:.1f}s")

  wiped /kaggle/temp/tartanimu
[train] 395 trajectories, 81931 windows, 16386200 frames, 333 MB, 15.96s, max |target - mean(vel_body)| = 0.00e+00
[val] 80 trajectories, 23714 windows, 4742800 frames, 96 MB, 4.26s, max |target - mean(vel_body)| = 0.00e+00
[test] 89 trajectories, 30644 windows, 6128800 frames, 74 MB, 2.51s
  windows   23.0s
splits_v1.json: verified, the draw still reproduces it
  splits    0.0s
[train] 2425 segments over 395 trajectories in 0.04s -> /kaggle/working/proj/mySolution/data/processed/cache/v1/segments/train.npz
[val] 617 segments over 80 trajectories in 0.01s -> /kaggle/working/proj/mySolution/data/processed/cache/v1/segments/val.npz
  segments  0.1s
  orientation (train) ...
  [train] 100/395 trajectories
  [train] 200/395 trajectories
  [train] 300/395 trajectories
[train] 16,386,200 frames in 56.38s -> gravity_body.npy
  gravity vs truth (train): 7.96 deg macro   {'car': 0.874, 'dog': 0.849, 'drone': 28.543, 'human': 1.561}   sign_ok=True
  orientation (va

## 6. Preflight

Resolves every run's config without training anything and checks the things
that would waste hours if they were wrong: the model reads the channels it
should, the device really is the GPU, the seeds really all moved, and no config
hash is already in `results.jsonl`. When the arm carries the gravity head, it
also pushes one real training batch through the real model and deletes the head
to prove no prediction moves; when it carries a control, it checks the control
is its reference with nothing but the device changed. When the arm is the
ground-truth oracle, it checks the three channels really are the true
down-direction, that the filter on the same frames sits a few degrees from them
(one sign and axis convention), and that the same config *without*
`run.diagnostic_only` is refused. Every arm keeps the sealed holdout shut.

The hashes will **not** match the laptop's. `run.device` is inside the hash on
purpose — the device changes the numerics, so a `cuda` run and an `mps` run of
the same science are not the same run.

In [6]:
import math

from src import config as config_mod
from src.prep.features import is_aligned, n_channels, needs_gravity
from src.utils import config_hash_appears


def _seeds(seed: int) -> dict:
    return {"seed_weights": seed, "seed_data": seed, "seed_augment": seed}


def overrides_for(seed: int) -> dict:
    """The arm: REFERENCE plus whatever this notebook moves."""
    over = {
        "run": {"name": f"{TAG}-seed{seed}", "milestone": MILESTONE, "device": DEVICE,
                "notes": NOTES, **_seeds(seed),
                # Stated only when set, like the gravity source below, so every
                # non-oracle arm keeps exactly the hash it had.
                **({"diagnostic_only": True} if DIAGNOSTIC_ONLY else {})},
        # Stated only when it moves (i.e. for a submission run), so an
        # ablation arm leaves it alone and keeps the reference config's own
        # hash. A submission run states it explicitly so the config diff -- and
        # the run.jsonl record -- says in plain sight that the seal was spent.
        "data": {"input_repr": INPUT_REPR,
                **({"exclude_seal": False} if not EXCLUDE_SEAL else {})},
        # Stated only when it moves, so an arm that leaves it alone keeps the
        # reference config's own hash and stays comparable across sessions.
        **({"prep": {"gravity": {"source": GRAVITY_SOURCE}}}
           if GRAVITY_SOURCE != "filter" else {}),
    }
    if GRAVITY_ESTIMATOR:
        # The one canonical spelling, every parameter written out, built by the
        # same function tests/test_m19.py uses -- so all three land on one hash.
        over = config_mod.deep_merge(
            over, config_mod.gravity_estimator_overrides(**GRAVITY_ESTIMATOR))
    if TRUTH_NOISE_DEG:
        # Same function tests/test_m20.py checks against M16's own hash at 0 deg.
        over = config_mod.deep_merge(
            over, config_mod.truth_gravity_overrides(TRUTH_NOISE_DEG, TRUTH_NOISE_TAU_S))
    if ATTITUDE_BRANCH:
        over = config_mod.deep_merge(
            over, config_mod.attitude_branch_overrides(**ATTITUDE_BRANCH))
    if TRAINVAL:
        # The same function tests/test_trainval.py checks, so both land on one hash.
        over = config_mod.deep_merge(over, config_mod.trainval_overrides(**TRAINVAL))
    if AUGMENT_RECIPE or TIMESCALE:
        # Written out in full -- the magnitudes, not just the recipe's name --
        # so the config hash and the run log both record exactly what was
        # applied, and a later recipe of the same name cannot be confused with
        # this one. The two ops share ONE `augment.ops` list, so it is built
        # here: deep_merge would let the second list replace the first. With a
        # single op this is exactly what that op's own override function returns.
        _aug = {"ops": []}
        _parts = ([config_mod.augment_overrides(AUGMENT_RECIPE)["augment"]] if AUGMENT_RECIPE else []) \
            + ([config_mod.timescale_overrides(**TIMESCALE)["augment"]] if TIMESCALE else [])
        for _part in _parts:
            _aug["ops"] = _aug["ops"] + list(_part["ops"])
            _aug.update({k: v for k, v in _part.items() if k != "ops"})
        over = config_mod.deep_merge(over, {"augment": _aug})
    if MODEL_OVERRIDES:
        over = config_mod.deep_merge(over, {"model": MODEL_OVERRIDES})
    if GRAVITY_HEAD:
        # The one canonical spelling of "head on, loss term appended", built by
        # the same function the command line and tests/test_m15.py use, so all
        # three land on the same hash.
        over = config_mod.deep_merge(
            over, config_mod.gravity_head_overrides(REFERENCE, GRAVITY_WEIGHT))
    if SMOKE_EPOCHS:
        over["optim"] = {"epochs": int(SMOKE_EPOCHS)}
    return over


def control_overrides_for(seed: int) -> dict:
    """The control: CONTROL_REFERENCE exactly, except for the device."""
    over = {"run": {"name": f"{CONTROL_TAG}-seed{seed}", "device": DEVICE,
                    "notes": CONTROL_NOTES, **_seeds(seed)}}
    if SMOKE_EPOCHS:
        over["optim"] = {"epochs": int(SMOKE_EPOCHS)}
    return over


# One job per run, in training order: the arm's own seeds first, because they
# are what this notebook exists for, then the control.
jobs = [(f"{TAG}-seed{s}", REFERENCE, overrides_for(s)) for s in SEEDS]
if RUN_CONTROL:
    jobs += [(f"{CONTROL_TAG}-seed{s}", CONTROL_REFERENCE, control_overrides_for(s))
             for s in CONTROL_SEEDS]

plan = []
for name, reference, over in jobs:
    cfg = config_mod.resolve(reference, over)
    done = config_hash_appears(cfg.hash)
    plan.append((name, reference, over, cfg, done))
    print(f"{name:<22} {reference:<9} #{cfg.hash}  "
          f"{'ALREADY DONE, will skip' if done else 'to run'}   diff={cfg.diff}")

ref = plan[0][3]
assert ref["data"]["input_repr"] == INPUT_REPR
want_channels = n_channels(INPUT_REPR, ref["data"]["extra_scalars"])
assert n_channels(ref["data"]["input_repr"], ref["data"]["extra_scalars"]) == want_channels
# `aligned` is six channels and `aligned_grav` nine, so the old "six means body"
# shortcut would have passed an arm with the wrong representation entirely.
# The gravity source is what this arm moves when it moves, so check the resolved
# config really carries it rather than trusting the override to have landed.
assert ref["prep"].get("gravity", {}).get("source", "filter") == GRAVITY_SOURCE
from src.prep.orientation import estimator_key, estimator_spec, orientation_path
assert estimator_spec(ref["prep"].get("gravity", {})) == estimator_spec(GRAVITY_ESTIMATOR or {}), \
    "GRAVITY_ESTIMATOR did not land in the resolved config"
GRAVITY_KEY = estimator_key(ref["prep"].get("gravity", {}))
if (GRAVITY_ESTIMATOR or {}).get("gyro_frame") == "truth":
    # Lever 3's oracle reads ground truth into the cache it trains on, so it must
    # carry the diagnostic stamp -- and the same config without it must refuse.
    from src.pipeline import build_dataset as _bd3
    assert DIAGNOSTIC_ONLY and ref["run"]["diagnostic_only"] is True
    _ungated = config_mod.resolve(
        REFERENCE, config_mod.gravity_estimator_overrides(**GRAVITY_ESTIMATOR))
    try:
        _bd3(_ungated, "val")
        raise AssertionError("the gyro-frame oracle built WITHOUT run.diagnostic_only")
    except ValueError as exc:
        assert "diagnostic_only" in str(exc), exc
    print("verified: without run.diagnostic_only the gyro-frame oracle is refused.")

# --- alignment preflight -------------------------------------------------------
# When the representation is one of the `aligned` pair, the model predicts in a
# gravity-aligned frame and `models.base.forward_batch` rotates the prediction
# back into the body frame. If that rotation were missing, dropped or transposed,
# nothing would crash: the submission would be a good prediction expressed in the
# wrong coordinates, and the only symptom would be a bad leaderboard score. So
# the round trip is proved here, on real labelled chunks, before any GPU time.
if is_aligned(INPUT_REPR):
    import torch
    from src.models.base import forward_batch
    from src.pipeline import build_dataset as _bd

    _ds = _bd(ref, "val")
    _b = _ds.batch(list(range(min(4, len(_ds)))))
    _R = _b["align_R"]
    assert tuple(_R.shape) == tuple(_b["v_gt"].shape[:2]) + (3, 3), "align_R has the wrong shape"
    _orth = (torch.einsum("bkij,bklj->bkil", _R, _R) - torch.eye(3)).abs().max().item()
    assert _orth < 1e-4, f"align_R is not orthonormal: {_orth:.2e}"

    class _Oracle(torch.nn.Module):
        def __init__(self, v):
            super().__init__(); self.v = v
        def forward(self, x, mask, cond):
            return {"velocity": self.v}

    _good = forward_batch(_Oracle(torch.einsum("bkij,bkj->bki", _R, _b["v_gt"])), _b)
    _err = (_good["velocity"] - _b["v_gt"]).abs().max().item()
    _bad = forward_batch(_Oracle(torch.einsum("bkji,bkj->bki", _R, _b["v_gt"])), _b)
    _bad_err = (_bad["velocity"] - _b["v_gt"]).abs().max().item()
    assert _err < 1e-4, f"ALIGNMENT GATE FAILED: round trip lost {_err:.2e} m/s"
    assert _bad_err > 1e-3, "a transposed rotation was not caught; the gate proves nothing"
    print(f"verified: perfect-in-aligned -> perfect-in-body to {_err:.1e} m/s, and a "
          f"transposed rotation is caught at {_bad_err:.2f} m/s.")
    del _ds, _b, _R, _good, _bad

# --- augmentation preflight ---------------------------------------------------
# Augmentation bugs are silent: nothing crashes if the input is rotated and the
# target is not, the model just learns a world that does not exist. So the gate
# from CLAUDE.md runs here, on this machine, on real labelled samples, before
# any GPU time is spent. A re-mounted sensor does not move the vehicle, so the
# augmented body velocity rotated into the world by the augmented attitude must
# equal the ORIGINAL world velocity, exactly.
from src.pipeline import build_dataset, build_training_dataset
from src.data.augment import quat_to_mat

_want_ops = (["remount"] if AUGMENT_RECIPE else []) + (["timescale"] if TIMESCALE else [])
assert (ref.get("augment", {}).get("ops") or []) == _want_ops, \
    "AUGMENT_RECIPE / TIMESCALE did not land in the resolved config"
if MODEL_OVERRIDES:
    def _landed(want, got, path="model"):
        for _k, _v in want.items():
            if isinstance(_v, dict):
                _landed(_v, got[_k], f"{path}.{_k}")
            else:
                assert got[_k] == _v, f"{path}.{_k} is {got[_k]!r}, want {_v!r}"
    _landed(MODEL_OVERRIDES, ref["model"])
    from src.models import build_model as _bm
    _mm = _bm(ref, in_channels=9 if needs_gravity(INPUT_REPR) else 6)
    print(f"verified: MODEL_OVERRIDES landed ({MODEL_OVERRIDES}); the model has "
          f"{_mm.n_parameters():,} parameters (m9_film with cnn_small: 1,378,279)")
    if ref["model"]["encoder"]["mode"] == "tcn":
        _enc = _mm.encoder if hasattr(_mm, "encoder") else None
        if _enc is not None and hasattr(_enc, "receptive_field"):
            assert _enc.receptive_field() >= 200, "the TCN cannot see a whole window"
            print(f"verified: TCN receptive field {_enc.receptive_field()} frames >= the 200 in a window")
    del _mm
_val = build_dataset(ref, "val")
assert _val.augment is None, "the val dataset was built augmented -- it must never be"
assert getattr(_val, "timescale", None) is None, "the val dataset was built time-scaled"
_tr = build_training_dataset(ref, verbose=False)
assert (_tr.augment is not None) == bool(AUGMENT_RECIPE), "training augmentation did not wire up"
assert (getattr(_tr, "timescale", None) is not None) == bool(TIMESCALE), \
    "training time scaling did not wire up"

# --- time-scaling preflight (M22) ---------------------------------------------
# The same silent-failure argument, sharper: a speed-up that scales gravity, uses
# s instead of s^2, or forgets the gyroscope crashes nothing. So the physics gate
# runs here on real ground truth -- a noise-free IMU synthesised from the recorded
# motion, sped up, against the same synthesis of the sped-up motion -- and two
# planted bugs must be caught, or the gate proves nothing. Then the training set
# itself is sampled: only drone chunks may change, the labels and gravity
# channels must be finite and physical, and the loader cost is measured.
if TIMESCALE:
    import numpy as np
    from src.data.timescale import GATE_LIMITS, check_gate, gate
    from src.prep.cache import load_cache as _load_cache
    from src.paths import WIN as _WIN

    # Gate at the largest drone speed-up this arm draws, as well as M22's own two.
    _smax = float(TIMESCALE.get("s_range", {}).get("drone", [1.0, 1.4])[1])
    _g = gate(_load_cache("train"), "drone", n_traj=6,
              scales=tuple(sorted({1.25, 1.4, _smax})))
    _fails = check_gate(_g)
    assert not _fails, f"TIME-SCALING GATE FAILED: {_fails}  {_g}"
    print(f"verified: time scaling matches ground-truth physics on 6 train drone flights -- "
          f"accel {_g['acc']:.4f} m/s2 (limit {GATE_LIMITS['acc']}), gyro {_g['gyr']:.5f} rad/s, "
          f"velocity {_g['velocity']:.4f} m/s; planted bugs read {_g['bug_linear_s']:.3f} m/s2 "
          f"and {_g['bug_gyro_unscaled']:.4f} rad/s, so the gate is sharp.")

    # With D9's train+val the training set is two parts; the train part is checked.
    _base = _tr.parts[0] if hasattr(_tr, "parts") else _tr
    _ts = _base.timescale
    _plain = build_dataset(ref, ref["data"]["train_split"],
                           traj_subset=sorted({s.traj_idx for s in _base.specs}), augment=False)
    assert [(s.traj_idx, s.start, s.length) for s in _plain.specs] == \
        [(s.traj_idx, s.start, s.length) for s in _base.specs]
    _rows = _base.cache.trajectories
    _plat = [str(_rows.platform.iloc[s.traj_idx]) for s in _base.specs]
    _drawn = [i for i, p in enumerate(_plat) if _ts.draw(i, p) is not None]
    _phased = [i for i, p in enumerate(_plat) if _ts.draw_phase(i, p)]
    assert _drawn or _phased, "nothing was drawn"
    assert all(_plat[i] in [q for q, v in _ts.p.items() if v > 0] for i in _drawn), \
        "a platform the recipe does not name was drawn"
    # Chunks with neither a speed-up nor a phase shift must be the plain data, bit for bit.
    _others = [i for i, p in enumerate(_plat)
               if _ts.draw(i, p) is None and not _ts.draw_phase(i, p)][:8]
    for _i in _others:
        _a, _b = _base[_i], _plain[_i]
        assert np.array_equal(_a["x"], _b["x"]) and np.array_equal(_a["v_gt"], _b["v_gt"]), \
            f"a {_plat[_i]} chunk changed under time scaling"
    if _phased:
        # M23: the re-cut path at shift 0 must BE the plain sample, on every platform;
        # at the drawn shift, the raw channels must be the recording's own frames.
        _pw = {}
        for _q in sorted(set(_plat[i] for i in _phased)):
            for _i in [i for i in _phased if _plat[i] == _q][:2]:
                _a0, _b = _base._timescaled_item(_i, 1.0, 0), _plain[_i]
                assert np.abs(_a0["x"] - _b["x"]).max() < 1e-4 and \
                    np.abs(_a0["v_gt"] - _b["v_gt"]).max() < 1e-5, f"{_q}: shift 0 is not the real sample"
                _k = _ts.draw_phase(_i, _q)
                _ak = _base._timescaled_item(_i, 1.0, _k)
                _s = _base.specs[_i]
                _lo = int(_rows.frame_offset.iloc[_s.traj_idx]) + _s.start * _WIN + _k
                _raw = (_ak["x"][:, :6, :].transpose(0, 2, 1).reshape(-1, 6)
                        * _base.norm.std[:6] + _base.norm.mean[:6])
                _ref = np.asarray(_base.cache.imu[_lo:_lo + len(_raw)], np.float64)
                assert np.abs(_raw - _ref).max() < 1e-3, f"{_q}: shifted channels are not the recording"
                _pw[_q] = _k
        print(f"verified: window phase -- {len(_phased)} of {len(_plat)} chunks re-cut this epoch; "
              f"shift 0 reproduces the real sample on {', '.join(sorted(_pw))}; shifted chunks are "
              f"the recording's own frames (e.g. " + ", ".join(f"{q} +{k}" for q, k in _pw.items())
              + " frames)")
    _t0 = time.perf_counter()
    _speeds, _lens = [], []
    for _i in _drawn[:12]:
        _a = _base[_i]
        _o = _plain[_i]
        _speeds.append(_a["speed"])
        _lens.append((_o["x"].shape[0], _a["x"].shape[0]))
        assert np.isfinite(_a["x"]).all() and np.isfinite(_a["v_gt"]).all()
        assert _a["x"].shape[1:] == _o["x"].shape[1:] and _a["x"].shape[0] <= _o["x"].shape[0]
        # the speed-up is visible in the target: mean speed up by about s
        _r = (np.linalg.norm(_a["v_gt"], axis=1).mean()
              / max(np.linalg.norm(_o["v_gt"], axis=1).mean(), 1e-6))
        assert 0.7 * _a["speed"] < _r < 1.4 * _a["speed"], \
            f"target speed moved {_r:.2f}x at s={_a['speed']:.2f}"
        if needs_gravity(INPUT_REPR):
            _gch = (_a["x"][:, 6:9, :].transpose(0, 2, 1).reshape(-1, 3)
                    * _base.norm.std[6:9] + _base.norm.mean[6:9])
            _nrm = np.linalg.norm(_gch, axis=1)
            assert np.abs(_nrm - 1.0).max() < 1e-3, "gravity channels are not unit vectors"
    _ms = 1e3 * (time.perf_counter() - _t0) / len(_speeds)
    _share = len(_drawn) / max(1, sum(p in _ts.p and _ts.p[p] > 0 for p in _plat))
    print(f"verified: {len(_drawn)} of the {sum(p == 'drone' for p in _plat)} drone chunks drawn "
          f"this epoch ({_share:.0%}); car/dog/human chunks bit-identical to the unaugmented set; "
          f"s in this sample {min(_speeds):.2f}-{max(_speeds):.2f}; flights shortened e.g. "
          + ", ".join(f"{a}->{b}" for a, b in _lens[:4]) + " windows")
    print(f"          loader cost {_ms:.0f} ms per time-scaled flight (first touch, NPZ load "
          f"included) -- about {_ms * 0.5 * 470 / 1e3:.0f} s of CPU per epoch at most")
    del _plain

if AUGMENT_RECIPE:
    import numpy as np
    from src.data.augment import WORLD_MIRROR

    _worst = 0.0
    for _i in range(0, len(_tr), max(1, len(_tr) // 25)):
        _a = _tr[_i]
        _q1, _v1 = _a["q_gt"].astype(np.float64), _a["v_gt"].astype(np.float64)
        _ok = np.linalg.norm(_q1, axis=1) > 1e-6
        if not _ok.any():
            continue
        _w1 = np.einsum("nij,nj->ni", quat_to_mat(_q1[_ok]), _v1[_ok])
        # the same windows, unaugmented, read straight from the cache
        _spec = _tr.specs[_i]
        _sl = _tr.cache.windows_slice(_spec.traj_idx, _spec.start, _spec.length)
        _q0 = np.asarray(_tr.cache.q_gt[_sl], np.float64)[_ok]
        _v0 = np.asarray(_tr.cache.v_gt[_sl], np.float64)[_ok]
        _w0 = np.einsum("nij,nj->ni", quat_to_mat(_q0), _v0)
        # A mirrored draw reflects the world too; a rotation leaves it alone.
        # Either is legal, so take whichever the chunk actually was.
        _err = min(float(np.abs(_w1 - _w0).max()),
                   float(np.abs(_w1 - _w0 @ WORLD_MIRROR.T).max()))
        _worst = max(_worst, _err)
    assert _worst < 1e-4, f"AUGMENTATION GATE FAILED: world velocity moved by {_worst:.2e} m/s"
    print(f"verified: augmentation preserves world velocity to {_worst:.1e} m/s "
          f"on real labelled chunks (the CLAUDE.md gate).")
    print(f"          recipe {AUGMENT_RECIPE!r}: yaw "
          + " ".join(f"{k}={v:g}deg" for k, v in sorted(_tr.augment.yaw_deg.items()))
          + f", tilt {_tr.augment.tilt_deg:g}deg")
del _val, _tr

# --- train+val preflight (D9) ---------------------------------------------------
# Selecting weights on data the model trained on would crash nothing -- it would
# just pick the epoch that memorised best and report a flattering score. So the
# training set is built here and checked chunk by chunk: every val chunk in it,
# the seal out of it (select_on "seal") or in it (select_on "none").
if TRAINVAL:
    from src.data.chunks import ConcatChunkDataset
    from src.pipeline import selection_set
    from src.prep.splits import load_splits

    assert selection_set(ref) == TRAINVAL["select_on"]
    assert ref["data"].get("train_extra") == ["val"], "val did not land in data.train_extra"
    _tr = build_training_dataset(ref, verbose=False)
    assert isinstance(_tr, ConcatChunkDataset), "the training set is not train + val"
    assert [p.cache.split for p in _tr.parts] == ["train", "val"], [p.cache.split for p in _tr.parts]
    _v = build_dataset(ref, "val")
    assert [(s.traj_idx, s.start, s.length) for s in _tr.parts[1].specs] == \
        [(s.traj_idx, s.start, s.length) for s in _v.specs], "not every val chunk trains"
    _sealed = set(load_splits().seal_rows(_tr.parts[0].cache))
    _in = {s.traj_idx for s in _tr.parts[0].specs}
    if TRAINVAL["select_on"] == "seal":
        assert not (_sealed & _in), "a sealed trajectory is in the training set"
    else:
        assert _sealed <= _in, "the final model must train on the seal too"
    # A mixed batch -- two train chunks, two val chunks -- through the real collate.
    _b = _tr.batch([0, 1, len(_tr) - 2, len(_tr) - 1])
    assert _b["x"].shape[2] == _tr.n_channels and bool(_b["mask"].any(dim=1).all())
    print(f"verified: training on {len(_in)} train trajectories "
          f"({'seal included' if not EXCLUDE_SEAL else 'seal held out'}) + {_v.cache.n_traj} val"
          f" -- {len(_tr):,} chunks, {len(_tr.parts[1]):,} of them val's; weights chosen on: "
          f"{TRAINVAL['select_on']}")
    del _tr, _v, _b

# The claim `GRAVITY_SOURCE = "chunk"` makes is that the gravity channels are a
# function of the chunk and nothing else. Build the val dataset and check it
# holds no trajectory-wide array at all -- the cache may well exist on disk, so
# "it worked" is not evidence; "it never opened it" is.
if needs_gravity(INPUT_REPR):
    from src.pipeline import build_dataset
    probe = build_dataset(ref, "val")
    if GRAVITY_SOURCE == "chunk":
        assert probe.gravity is None, "chunk mode must not hold a trajectory-wide array"
        print("verified: the val dataset holds no trajectory-wide gravity array.")
        print("          The three channels come from each chunk's own frames, "
              "inside the loader.")
    elif GRAVITY_SOURCE == "truth":
        import numpy as np
        # FS, not DT: DT is the scorer's 1.0 s per WINDOW; the filter steps per
        # FRAME. Passing DT integrates the gyroscope 200x too fast and reads
        # 60-90 deg here -- which is how this line learned the difference.
        from src.paths import FS, PLATFORMS, WIN
        from src.prep.orientation import gravity_direction, gravity_from_quaternion

        assert probe.gravity is None, "the oracle must not read the filter cache"
        assert ref["run"]["diagnostic_only"] is True
        # One full chunk per platform: the channels, un-normalised, must BE the
        # truth; and the chunk filter on the same frames must land a few degrees
        # from it -- the same convention the normalisation constants were
        # measured on. A flipped sign or swapped axis reads ~90-180 deg here.
        agree, noise = {}, []
        for p in PLATFORMS:
            j = next(k for k, s in enumerate(probe.specs)
                     if str(probe.cache.trajectories.platform.iloc[s.traj_idx]) == p)
            s = probe.specs[j]
            lo = int(probe.cache.trajectories.frame_offset.iloc[s.traj_idx]) + s.start * WIN
            hi = lo + s.length * WIN
            want = gravity_from_quaternion(np.asarray(probe.cache.quat_gt[lo:hi], np.float64))
            got = (probe[j]["x"][:, 6:9, :].transpose(0, 2, 1).reshape(-1, 3)
                   * probe.norm.std[6:9] + probe.norm.mean[6:9])
            if TRUTH_NOISE_DEG:
                g_ = got / np.linalg.norm(got, axis=1, keepdims=True)
                noise.append(np.degrees(np.arccos(np.clip((g_ * want).sum(1), -1.0, 1.0))))
            else:
                assert np.abs(got - want).max() < 1e-4, f"{p}: channels are not the truth"
            frames = np.asarray(probe.cache.frames(s.traj_idx, s.start, s.length), np.float64)
            filt = gravity_direction(frames[:, :6], 1.0 / FS)
            agree[p] = float(np.median(np.degrees(np.arccos(
                np.clip((filt * want).sum(1), -1.0, 1.0)))))
        assert max(agree.values()) < 10.0, f"truth and filter disagree: {agree}"
        if TRUTH_NOISE_DEG:
            # Four chunks are a small sample of a process that drifts over 3 s,
            # so the bar is loose -- it exists to catch 0 deg or 90 deg, not 10%.
            _med = float(np.median(np.concatenate(noise)))
            assert 0.6 * TRUTH_NOISE_DEG < _med < 1.4 * TRUTH_NOISE_DEG, \
                f"the degraded channels sit {_med:.2f} deg from the truth, asked {TRUTH_NOISE_DEG}"
            print(f"verified: the channels are the truth plus a drifting error, median "
                  f"{_med:.2f} deg on four probe chunks (asked {TRUTH_NOISE_DEG}); the filter "
                  "sits " + ", ".join(f"{p} {v:.1f}" for p, v in agree.items())
                  + " deg from the truth -- one convention.")
        else:
            print("verified: the three channels ARE the ground-truth down-direction, and the "
                  "filter on the same frames sits "
                  + ", ".join(f"{p} {v:.1f}" for p, v in agree.items())
                  + " deg from them -- one convention.")
        # The gate itself: the same config minus the flag must be refused.
        _ungated = config_mod.resolve(REFERENCE, {"prep": {"gravity": {"source": "truth"}}})
        try:
            build_dataset(_ungated, "val")
            raise AssertionError("the oracle built WITHOUT run.diagnostic_only -- the gate is open")
        except ValueError as exc:
            assert "diagnostic_only" in str(exc), exc
        print("verified: without run.diagnostic_only the oracle is refused.")
    else:
        assert probe.gravity is not None
        # The dataset must open THIS arm's estimator cache -- a two-sided arm
        # silently reading the default filter's file would train the control.
        _want = orientation_path("val", key=GRAVITY_KEY)
        assert Path(probe.gravity.filename).resolve() == _want.resolve(), \
            f"the val dataset opened {probe.gravity.filename}, expected {_want}"
        print(f"verified: the val dataset reads {_want.name}")
    del probe
# Only the D9 seal run opens the seal: it is the set that run chooses weights on.
assert bool(ref["eval"].get("seal")) == bool(TRAINVAL and TRAINVAL["select_on"] == "seal"), \
    "the sealed holdout stays shut for this arm"
assert ref["data"].get("exclude_seal", True) == EXCLUDE_SEAL,     f"exclude_seal resolved to {ref['data'].get('exclude_seal', True)}, expected {EXCLUDE_SEAL}"
if not EXCLUDE_SEAL:
    print("verified: this run trains on the sealed holdout too (data.exclude_seal=False) -- "
          + ("a submission run, not an ablation." if SUBMIT_TEST else
             "trained like the submission runs so its val compares with theirs; "
             "a diagnostic, no test file."))
if SUBMIT_TEST:
    assert BUILD_TEST_CACHE, "SUBMIT_TEST needs BUILD_TEST_CACHE=True to have a test cache to predict"
    assert len(SEEDS) == 1,         f"SUBMIT_TEST expects exactly one seed (one shared set of weights), got {SEEDS}"
assert bool(ref["run"]["diagnostic_only"]) == DIAGNOSTIC_ONLY
assert ref["run"]["device"] == DEVICE
assert ref["model"]["context"]["mode"] == "bigru"
assert ref["model"]["conditioner"]["mode"] == "film"
assert ref["optim"]["epochs"] == (SMOKE_EPOCHS or (TRAINVAL or {}).get("epochs", 150))
assert bool(ref["model"]["heads"].get("gravity", False)) == GRAVITY_HEAD
assert (ref["model"].get("attitude") or None) == (ATTITUDE_BRANCH or None), \
    "ATTITUDE_BRANCH did not land in the resolved config"

if ATTITUDE_BRANCH:
    # Build the real model on the CPU and check the wiring before GPU time: the
    # encoder reads six channels in "late" and nine in "both", and the branch's
    # output layer starts at exactly zero (so the run starts from m9_film's
    # architecture minus, or plus, the branch -- never from a random offset).
    import torch
    from src.models import build_model as _bm
    torch.manual_seed(0)
    _m = _bm(ref, in_channels=n_channels(ref["data"]["input_repr"], ref["data"]["extra_scalars"]))
    _first = next(p for _, p in _m.encoder.named_parameters() if p.dim() == 3)
    _want = 6 if ATTITUDE_BRANCH["mode"] == "late" else 9
    assert _first.shape[1] == _want, f"encoder reads {_first.shape[1]} channels, want {_want}"
    assert _m.attitude is not None and float(_m.attitude.out.weight.abs().sum()) == 0.0
    print(f"verified: attitude branch '{ATTITUDE_BRANCH['mode']}' -- encoder reads {_want} "
          f"channels, branch output starts at zero, {_m.n_parameters():,} parameters")
    del _m

if GRAVITY_HEAD:
    import torch
    from src.losses import build_loss, gravity_valid
    from src.models import build_model, forward_batch
    from src.pipeline import build_training_dataset

    g_terms = [t for t in ref["loss"]["terms"]
               if t["name"] == "gravity" and t.get("enabled", True)]
    assert len(g_terms) == 1 and g_terms[0]["weight"] == GRAVITY_WEIGHT, ref["loss"]["terms"]
    # The head is read against the three six-channel Kaggle runs, so its
    # reference must still be exactly their science: seed 42 on a GPU, with
    # none of this arm's changes, must land on the m7-body seed-42 hash.
    if REFERENCE == "m7_body":
        base = config_mod.resolve("m7_body", {"run": {"device": "cuda"}}).hash
        assert base == "bb699b34", f"m7_body resolves to #{base}, the runs were #bb699b34"
        print("verified: m7_body is still the science of the m7-body runs (#bb699b34).")

    # One real training batch through the real model, on the GPU. The target is
    # built from q_gt on the device, every real train window must carry one, and
    # the loss must come out finite.
    probe_ds = build_training_dataset(ref, verbose=False)
    torch.manual_seed(0)
    model = build_model(ref, in_channels=probe_ds.n_channels).to(DEVICE)
    idx = list(range(int(ref["data"]["batch_size"])))
    batch = {k: (v.to(DEVICE) if torch.is_tensor(v) else v)
             for k, v in probe_ds.batch(idx).items()}
    _, parts = build_loss(ref)(forward_batch(model, batch), batch)
    n_real = int(batch["mask"].sum())
    n_valid = int(gravity_valid(batch["q_gt"], batch["mask"]).sum())
    assert n_valid == n_real > 0, f"only {n_valid} of {n_real} real windows carry a quaternion"
    assert all(math.isfinite(parts[k]) for k in ("gravity", "gravity_angle_deg", "total")), parts
    n_head = sum(p.numel() for p in model.gravity_head.parameters())

    # Delete-the-head, on the CPU where every kernel is deterministic: the head
    # exists only for its gradient, so removing it must move no prediction.
    cpu = model.cpu().eval()
    cpu_b = {k: (v.cpu() if torch.is_tensor(v) else v) for k, v in batch.items()}
    with torch.no_grad():
        v_with = cpu(cpu_b["x"], cpu_b["mask"], cpu_b["cond"])["velocity"]
        cpu.gravity_head = None
        v_without = cpu(cpu_b["x"], cpu_b["mask"], cpu_b["cond"])
    assert "gravity" not in v_without
    assert torch.equal(v_with, v_without["velocity"]), "deleting the gravity head moved a prediction"
    print(f"verified: gravity head {n_head:,} parameters; {n_valid}/{n_real} windows of a real "
          f"batch supervised; untrained angle {parts['gravity_angle_deg']:.1f} deg "
          f"(a random guess is ~90); deleting the head moves no prediction.")
    del probe_ds, model, cpu, batch, cpu_b

if RUN_CONTROL:
    allowed = {"run.name", "run.notes", "run.device", "optim.epochs",
               "run.seed_weights", "run.seed_data", "run.seed_augment"}
    for name, reference, over, cfg, _ in plan[len(SEEDS):]:
        moved = set(cfg.diff) - allowed
        assert not moved, f"{name} must be {reference} unchanged; it also moves {moved}"
        assert cfg["run"]["device"] == DEVICE
    print(f"verified: the control is {CONTROL_REFERENCE} with only the device moved "
          f"({DEVICE}).")

print(f"\nOK: {n_channels(ref['data']['input_repr'], ref['data']['extra_scalars'])} input "
      f"channels from {INPUT_REPR!r}, gravity from {GRAVITY_SOURCE!r} "
      f"(estimator {GRAVITY_KEY or 'cf, the default'}), "
      f"gravity head {'ON' if GRAVITY_HEAD else 'off'}, "
      f"{'DIAGNOSTIC ONLY, ' if DIAGNOSTIC_ONLY else ''}"
      f"{ref['optim']['epochs']} epochs, chunk_len {ref['data']['chunk_len']}, "
      f"batch_size {ref['data']['batch_size']}, device {ref['run']['device']}, "
      f"{sum(not d for *_, d in plan)} run(s) to train")

m23-phase-seed42       m9_film   #d8c05965  to run   diff={'run.name': 'm23-phase-seed42', 'run.notes': 'M23: M22 (timescale_drone, p 0.5, s 1.0-1.4) + window-phase jitter (grid shifted 1-199 frames, all platforms, every chunk). SUBMISSION run: one seed, trained on train+seal, predicts the test split.', 'run.milestone': 'M23', 'run.device': 'cuda', 'data.exclude_seal': False, 'augment.ops': ['timescale'], 'augment.timescale': {'recipe': 'timescale_drone', 's_range': {'car': [1.0, 1.0], 'dog': [1.0, 1.0], 'drone': [1.0, 1.4], 'human': [1.0, 1.0]}, 'p': {'car': 0.0, 'dog': 0.0, 'drone': 0.5, 'human': 0.0}, 'fc_hz': 8.0, 'split_box_seconds': 5.0, 'phase_p': {'car': 1.0, 'dog': 1.0, 'drone': 1.0, 'human': 1.0}}}
verified: time scaling matches ground-truth physics on 6 train drone flights -- accel 0.0123 m/s2 (limit 0.15), gyro 0.00189 rad/s, velocity 0.0035 m/s; planted bugs read 0.515 m/s2 and 0.1737 rad/s, so the gate is sharp.
verified: window phase -- 1403 of 1403 chunks re-cut this ep

## 7. Train

One `pipeline.run` per job — the same function the command line calls, so a
Kaggle run and a laptop run are the same run and land in the same
`results.jsonl` schema.

**Measured on the M7 runs: 19 to 31 minutes per seed on a T4**, against 2.2 to
3.8 hours on the laptop, so a three-seed arm plus a control fits in about two
hours of a twelve-hour session. The batch is small on purpose (4 chunks x 64
windows = the same 256 window slots per step as M5). Do not raise `batch_size`
to feed the GPU better: it would make this a batch-size study as well.

A run that fails does not stop the others. The loop is resumable — re-running
this cell skips whatever already landed. `history.json` is rewritten in the run
directory after every epoch, so an interrupted run is still readable. With the
gravity head on, every epoch line also carries the train and val gravity angle.

**About the progress output.** The training loop redraws a one-line step
counter four times a second by overwriting itself with a carriage return. A
terminal shows that as a single moving line; a notebook *keeps every redraw*, so
hours of it would be hundreds of thousands of writes and a saved notebook too
large to open. `Heartbeat` below lets one step line through every
`HEARTBEAT_SECONDS` and converts it into an ordinary line. Epoch summaries,
scores and errors are never throttled.

In [7]:
import contextlib, io as _io

from src import pipeline


class Heartbeat(_io.TextIOBase):
    """Pass everything through, but rate-limit the in-place step counter.

    The training loop writes its progress line as `\r  epoch N  step i/M ...`
    with no newline, four times a second. Only those are throttled, and the one
    that survives is turned into a normal line so the Kaggle log stays readable
    and scrollable. Anything else -- epoch summaries, the val score, the
    per-platform table, tracebacks -- is forwarded untouched and immediately.
    """

    def __init__(self, inner, period: float):
        self.inner, self.period, self.last = inner, float(period), 0.0

    def write(self, s: str) -> int:
        if s.startswith("\r") and " step " in s and not s.endswith("\n"):
            now = time.time()
            if now - self.last < self.period:
                return len(s)
            self.last = now
            s = s.lstrip("\r") + "\n"
        return self.inner.write(s)

    def flush(self) -> None:
        self.inner.flush()


results = {}
for name, reference, over, cfg, already_done in plan:
    if already_done:
        print(f"\n=== {name}: skipped, #{cfg.hash} already in results.jsonl ===")
        continue
    print(f"\n{'=' * 70}\n=== {name}  {reference}  #{cfg.hash}  "
          f"{time.strftime('%H:%M:%S')} ===\n{'=' * 70}", flush=True)
    started = time.perf_counter()
    try:
        with contextlib.redirect_stdout(Heartbeat(sys.stdout, HEARTBEAT_SECONDS)):
            record = pipeline.run(reference=reference, overrides=over)
        results[name] = record
        m = record["metrics"]
        ga = record.get("gravity_angle_deg")
        if m is None:                             # D9: val trained, so it was not scored
            s = record.get("seal")
            print(f"\n>>> {name}: "
                  + (f"SEAL score {s['score']:.4f}  AVE {s['macro_ave']:.4f}  ATE20 "
                     f"{s['macro_ate20']:.4f}  |  AVE "
                     + "  ".join(f"{p} {v['ave']:.3f}" for p, v in s["per_platform"].items())
                     if s else "nothing held out -- final EMA weights kept, no local score")
                  + f"  ({(time.perf_counter() - started) / 3600:.2f} h)", flush=True)
            continue
        print(f"\n>>> {name}: score {m['score']:.4f}  "
              f"AVE {m['macro_ave']:.4f}  ATE20 {m['macro_ate20']:.4f}  |  AVE "
              + "  ".join(f"{p} {v['ave']:.3f}" for p, v in m["per_platform"].items())
              + (f"  |  gravity {ga['macro_median']:.1f} deg" if ga else "")
              + f"  ({(time.perf_counter() - started) / 3600:.2f} h)", flush=True)
    except Exception as exc:                      # one bad run must not cost the others
        import traceback; traceback.print_exc()
        print(f"\n>>> {name} FAILED: {type(exc).__name__}: {exc}", flush=True)


=== m23-phase-seed42  m9_film  #d8c05965  07:51:16 ===
time scaling: drone p=0.5 s=1..1.4  vibration kept above 8 Hz  seed=42
window phase (M23): car p=1  dog p=1  drone p=1  human p=1  -- window grid shifted 1-199 frames, real frames and real labels
training: 1,403 chunks, 470 steps/epoch, 150 epochs, 1,378,279 parameters
platform share per batch: 0=0.250  1=0.250  2=0.250  3=0.250
  epoch   0  step    1/470  loss 1.2868    0.2 steps/s
  epoch   0  step   63/470  loss 0.9907    1.8 steps/s
  epoch   0   68.2s  loss 0.6613  ave 0.7146  platform_acc 0.722  |  val[live] 0.8600  val[ema] 1.8522   <- best
  epoch   1  step    1/470  loss 0.5074   10.4 steps/s
  epoch   1   28.4s  loss 0.4805  ave 0.5581  platform_acc 0.936  |  val[live] 0.8307  val[ema] 1.1553   <- best
  epoch   2  step   29/470  loss 0.4519   16.5 steps/s
  epoch   2   28.7s  loss 0.4053  ave 0.4790  platform_acc 0.962  |  val[live] 0.6977  val[ema] 0.8554   <- best
  epoch   3  step   58/470  loss 0.4348   19.0 steps/s

## 8. Generate the Kaggle test submission

Only runs when `SUBMIT_TEST = True`. `pipeline.run()` above already trained the
model and, in the same call, scored it on **val** (never on test -- val is
where the best epoch is chosen and where a local score is possible at all,
since test carries no labels). This cell reloads each job's saved checkpoint
--- no retraining --- and runs one more forward pass over the **test** split,
then writes `submission_test.csv` next to that run's `submission_val.csv`.

The checkpoint's own directory is found by globbing `runs/` for
`*-<run name>-<config hash>` rather than recomputed from `utils.run_dir`,
because that function stamps *today's* date -- a job that was already done in
an earlier session (the resumable-loop skip above) lives under an earlier
date, and recomputing the path would silently point at a directory that was
never created.

In [8]:
if not SUBMIT_TEST:
    print("SUBMIT_TEST is False -- this arm does not produce a test submission.")
else:
    import shutil as _shutil

    import torch

    from src.models import build_model
    from src.pipeline import predict_split
    from src.post.submission import write_submission
    from src.prep.features import n_channels as _n_channels

    def _find_run_dir(name: str, config_hash: str) -> Path:
        matches = sorted(RUNS.glob(f"*-{name}-{config_hash}"))
        if not matches:
            raise SystemExit(f"no run directory for {name} #{config_hash} -- "
                             f"did the training cell (## 7) run first?")
        return matches[-1]

    submission_paths = []
    for name, reference, over, cfg, _ in plan:
        out_dir = _find_run_dir(cfg["run"]["name"], cfg.hash)
        ckpt_path = out_dir / "model.pt"
        if not ckpt_path.is_file():
            print(f"!! {name}: no checkpoint at {ckpt_path} -- did training finish? skipping.")
            continue

        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model = build_model(cfg, in_channels=_n_channels(
            cfg["data"]["input_repr"], cfg["data"]["extra_scalars"]))
        model.load_state_dict(ckpt["state_dict"])
        model.to(DEVICE).eval()

        table, cache, stats = predict_split(cfg, "test", verbose=True, model=model)
        sub_path = write_submission(
            table, out_dir / "predictions" / "submission_test.csv",
            cache.window_id, "test")
        submission_paths.append(sub_path)
        print(f">>> {name}: wrote {sub_path}  ({len(table)} rows)")

    if submission_paths:
        top = Path("/kaggle/working/submission.csv")
        _shutil.copy(submission_paths[-1], top)
        print(f"\nready: {top}")
        print("Download it from the Output panel (or take it from the runs/ zip "
              "in the next cell) and upload it to the competition's submission page.")
    else:
        print("\nno checkpoint was found for any job -- nothing written.")

  batch 100/129
>>> m23-phase-seed42: wrote /kaggle/working/proj/mySolution/runs/20260919-m23-phase-seed42-d8c05965/predictions/submission_test.csv  (30644 rows)

ready: /kaggle/working/submission.csv
Download it from the Output panel (or take it from the runs/ zip in the next cell) and upload it to the competition's submission page.


## 9. The tables

Read back out of `runs/results.jsonl` by `src/report.py`, never assembled by
hand — these are the report's ablation tables.

The spread table is the number this notebook exists to produce: the mean and
2σ over the three seeds, read against the comparison points printed under it.
A difference smaller than the noise floor did not happen.

In [9]:
from src.report import load_runs, reference_points, runs_table, spread_table

COMPARE = """the control is M22, the SAME config without the phase, already run:\n    m22-timescale-seed42   val 0.2288  ->  public 0.36871\n    (and m9-film-final-seed42, no augmentation: val 0.2421 -> 0.39119)\n    val noise floor, one seed vs one seed: 2sigma ~ 0.0086\n    PREDICTED: val better, drone most; the public board decides."""

print(reference_points())
print()
print(runs_table(load_runs()))
print()
# spread_table takes the WHOLE log and a run-name prefix: it uses the prefix
# only to find the arm, then regroups by resolved configuration, so a smoke run
# can never be averaged in with a real one.
if load_runs(name_prefix=TAG):
    print(spread_table(load_runs(), TAG))
    print("\n" + COMPARE)
else:
    print(f"no val-scored run named {TAG}* in results.jsonl yet.")
if RUN_CONTROL and load_runs(name_prefix=CONTROL_TAG):
    print(f"\ncontrol, {CONTROL_REFERENCE} on this GPU:")
    print(runs_table(load_runs(name_prefix=CONTROL_TAG)))

reference points on val
  0.0128  ground-truth velocities, the floor
  0.4185  released baseline routed by true platform -- a TARGET, illegal to submit
  0.8346  released baseline with no routing -- the bar a legal model must beat
  1.0144  all zeros

run                  ms          score    seal     car     dog   drone   human  diff
------------------------------------------------------------------------------------
m23-phase-seed42     M23        0.2248       -   0.131   0.098   0.382   0.083  augment.ops=['timescale'], augment.timescale={'fc_hz': 8.0, 'p': {'car': 0.0, 'dog': 0.0, 'drone': 0.5, 'human': 0.0}, 'phase_p': {'car': 1.0, 'dog': 1.0, 'drone': 1.0, 'human': 1.0}, 'recipe': 'timescale_drone', 's_range': {'car': [1.0, 1.0], 'dog': [1.0, 1.0], 'drone': [1.0, 1.4], 'human': [1.0, 1.0]}, 'split_box_seconds': 5.0}, data.exclude_seal=False

seed spread for 'm23-phase': only 1 run(s) share this configuration; a spread needs at least two.

the control is M22, the SAME config witho

## 9b. The M22 readout, read by the rule written before the run

**Why not just the val score.** Test's drones move harder than train's (linear
acceleration 1.87×) but val's move more gently (0.75×), so val's plain drone AVE
under-rewards exactly what this arm targets. This cell splits val's **drone**
windows at their median intensity (the RMS of linear acceleration below 5 Hz,
IMU-only, `src/evaluation/intensity.py`) and scores each half the way the metric
scores a platform: per window, per flight, then over flights.

**The control** (`m9-film-final-seed42`, 150 epochs, same seed, no time scaling):
val 0.2421, public **0.39119**; drone AVE 0.4271 overall, **0.3401 calm half,
0.4616 hard half**. Across four `m9_film` runs each half has a seed sd of 0.0067,
so one run against one run must differ by more than **0.019** to count.

**The rule, fixed before the run.**

1. **Hard half of val drone** is the primary local signal: better than 0.4616 by
   more than 0.019 = *helps*; worse by more than 0.019 = *hurts*; otherwise the
   leaderboard decides.
2. **Car, dog and human are untouched by this arm**, so a move beyond 0.0075 m/s
   (what one retrain moved an untouched platform, finding 23) is noise, flagged.
3. **Overall val** is expected flat or slightly worse; 2σ for one seed against
   one seed is 0.0086.
4. **The public leaderboard decides.** Submit `submission_test.csv`; compare with
   0.39119. The leaderboard's own seed noise is still unmeasured (DIAGNOSIS D12).

In [10]:
from pathlib import Path

import numpy as np

from src.evaluation.intensity import halves, window_intensity
from src.paths import PLATFORMS
from src.report import load_runs

# Measured before the run -- .claude/agentTests/2026-09-19_time-scaling-design/readout_references.out
CONTROL_VAL, CONTROL_PUBLIC = 0.2421, 0.39119
CONTROL_AVE = {"car": 0.1364, "dog": 0.1068, "drone": 0.4271, "human": 0.0891}
CONTROL_DRONE = {"calm": 0.3401, "intense": 0.4616}
ONE_VS_ONE = 0.019       # 2 sigma of a one-seed difference on either drone half
RETRAIN = 0.0075         # what one retrain moved an untouched platform (finding 23)
VAL_FLOOR = 0.0086       # 2 sigma, one seed vs one seed, whole val score

rows = load_runs(name_prefix=TAG)
if not rows:
    print(f"no finished {TAG}* run yet -- nothing to read.")
else:
    table = window_intensity("val", "drone")
    print(f"val drone: {len(table)} windows, {table.traj_id.nunique()} flights, split at "
          f"{table.intensity.median():.3f} m/s2 of linear acceleration")
for r in rows:
    m = r["metrics"]
    smoke = "optim.epochs" in (r.get("config_diff") or {})
    h = halves(Path(r["run_dir"]) / "predictions" / "submission_val.csv", table)
    print(f"\n{r['run_name']}  #{r['config_hash']}"
          + ("   ** REHEARSAL (changed optim.epochs): numbers only, no verdict **" if smoke else ""))
    d = m["score"] - CONTROL_VAL
    print(f"  val score {m['score']:.4f}  vs control {CONTROL_VAL}  ->  {d:+.4f}   "
          + ("inside the one-seed floor" if abs(d) <= VAL_FLOOR else
             "better beyond the floor" if d < 0 else "worse beyond the floor"))
    for p in PLATFORMS:
        a = m["per_platform"][p]["ave"]
        note = ""
        if p != "drone" and abs(a - CONTROL_AVE[p]) > RETRAIN:
            note = "   <- untouched platform moved more than a retrain does; noise, but look"
        print(f"  {p:<6} AVE {a:.4f}  ({a - CONTROL_AVE[p]:+.4f}){note}")
    dc = h["calm"] - CONTROL_DRONE["calm"]
    di = h["intense"] - CONTROL_DRONE["intense"]
    print(f"  drone, calm half {h['calm']:.4f} ({dc:+.4f})   HARD half {h['intense']:.4f} ({di:+.4f})"
          f"   [control {CONTROL_DRONE['calm']} / {CONTROL_DRONE['intense']}]")
    if smoke:
        continue
    if di < -ONE_VS_ONE:
        verdict = "HELPS on the hard half of val drone, beyond one seed's noise"
    elif di > ONE_VS_ONE:
        verdict = "HURTS on the hard half of val drone, beyond one seed's noise"
    else:
        verdict = f"inside +/-{ONE_VS_ONE} on the hard half: val cannot tell"
    print(f"\n==> {verdict}. Submit predictions/submission_test.csv and compare its public "
          f"score with {CONTROL_PUBLIC} -- the leaderboard decides.")
    # Added 2026-09-19, after M22 itself ran: the bar a stronger dose must beat.
    print(f"  second reference, M22 at s 1.0-1.4 p 0.5 (#7d587343): val 0.2288, public 0.36871, "
          f"drone calm 0.2965 / HARD 0.4045  ->  this run's hard half {h['intense'] - 0.4045:+.4f}")

val drone: 2327 windows, 48 flights, split at 1.326 m/s2 of linear acceleration

m23-phase-seed42  #d8c05965
  val score 0.2248  vs control 0.2421  ->  -0.0173   better beyond the floor
  car    AVE 0.1309  (-0.0055)
  dog    AVE 0.0983  (-0.0085)   <- untouched platform moved more than a retrain does; noise, but look
  drone  AVE 0.3818  (-0.0453)
  human  AVE 0.0830  (-0.0061)
  drone, calm half 0.3051 (-0.0350)   HARD half 0.3994 (-0.0622)   [control 0.3401 / 0.4616]

==> HELPS on the hard half of val drone, beyond one seed's noise. Submit predictions/submission_test.csv and compare its public score with 0.39119 -- the leaderboard decides.
  second reference, M22 at s 1.0-1.4 p 0.5 (#7d587343): val 0.2288, public 0.36871, drone calm 0.2965 / HARD 0.4045  ->  this run's hard half -0.0051


## 10. Pack it up

Zips the whole `runs/` tree — run directories and `results.jsonl` — into
`/kaggle/working/tartanimu_runs.zip`. Download it from the right-hand *Output*
panel, or *Save Version* and take it from the version's output.

Locally: unzip, move the run directories into `mySolution/runs/`, and append the
Kaggle `results.jsonl` lines to the local file. Then
`.venv/bin/python -m src.report --spread <TAG>` reads them like any other run.

In [11]:
import zipfile

zip_path = Path("/kaggle/working/tartanimu_runs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(RUNS.rglob("*")):
        if f.is_file():
            z.write(f, f.relative_to(RUNS.parent))

print(f"{zip_path}  ({zip_path.stat().st_size / 1e6:.1f} MB)\n")
for f in sorted(RUNS.rglob("*")):
    if f.is_file():
        print(f"  {str(f.relative_to(RUNS)):<58} {f.stat().st_size / 1e6:>8.2f} MB")

/kaggle/working/tartanimu_runs.zip  (6.8 MB)

  20260919-m23-phase-seed42-d8c05965/config.json                 0.00 MB
  20260919-m23-phase-seed42-d8c05965/env.json                    0.00 MB
  20260919-m23-phase-seed42-d8c05965/history.json                0.10 MB
  20260919-m23-phase-seed42-d8c05965/metrics.json                0.00 MB
  20260919-m23-phase-seed42-d8c05965/model.pt                    5.53 MB
  20260919-m23-phase-seed42-d8c05965/predictions/submission_test.csv     2.13 MB
  20260919-m23-phase-seed42-d8c05965/predictions/submission_val.csv     1.63 MB
  results.jsonl                                                  0.00 MB
